# Five molecule provenance-gap lineage rebuild

This task-scoped notebook calls the real immutable builder for `molecule_associated_phenotype`, `molecule_contraindicates_disease`, `molecule_parent_of_molecule`, `molecule_synergizes_molecule`, and the original `molecule_treats_disease` edge lineage. It never writes canonical tables.

The accepted flattened source boundary is TxGNN/DeepPurpose Dataverse DOI `10.7910/DVN/CNQV69`, v6.0 `kg.csv`, file ID `7144484`, MD5 `aac8191d4fbc5bf09cdf8c3c78b4e75f`, CC0-1.0. Constituent upstream release labels are not encoded in the flattened file and are not invented here.

## Safety and execution contract

Fixture replay is bounded and local. Full fetch/replay/parity is heavy and must run on `txgnn-worker` after approved lifecycle admission, with the exact source checksum and a read-only canonical snapshot. Outputs are create-only under `artifacts/staged/t_86299745/` or `gs://jouvencekb/staging/t_86299745/`. No canonical promotion is available from this notebook.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from manage_db.rebuild_molecule_provenance_gaps import TXGNN_DATASET, validate_launcher_receipt, verify_replay, write_replay

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'pyproject.toml').exists() and (REPO_ROOT.parent / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent

FIXTURE_REPLAY = os.environ.get('TXGNN_MOLECULE_GAP_FIXTURE_REPLAY', '0') == '1'
FULL_REPLAY = os.environ.get('TXGNN_MOLECULE_GAP_FULL_REPLAY', '0') == '1'
ALLOW_CANONICAL_WRITES = False
assert not ALLOW_CANONICAL_WRITES
TXGNN_DATASET

## Bounded fixture replay

Set `TXGNN_MOLECULE_GAP_FIXTURE_REPLAY=1` to execute the checked-in fixture through the same acquisition identity, mapping, quarantine, edge and evidence writer used by full replay.

In [ ]:
if FIXTURE_REPLAY:
    write_replay(
        REPO_ROOT / 'tests/fixtures/molecule_provenance_gaps/txgnn_kg_fixture.csv',
        REPO_ROOT / 'artifacts/staged/t_86299745/notebook-fixture-replay',
        fixture=True,
        fixture_allowed_root=REPO_ROOT / 'artifacts/staged/t_86299745',
        canonical_dir=None,
        chunksize=2,
    )
else:
    print('SKIPPED: set TXGNN_MOLECULE_GAP_FIXTURE_REPLAY=1 for bounded replay')

## Full worker replay and parity

After lifecycle admission on `txgnn-worker`, materialize Dataverse file 7144484 once, verify size 981751236 and MD5 `aac8191d4fbc5bf09cdf8c3c78b4e75f`, copy the five frozen canonical edge generations and node/evidence inputs into a read-only local snapshot, then set `TXGNN_MOLECULE_GAP_FULL_REPLAY=1`. The manifest and parity report remain staged-only and review-required.

In [ ]:
if FULL_REPLAY:
    admission = validate_launcher_receipt(Path(os.environ['TXGNN_MOLECULE_GAP_LAUNCHER_RECEIPT']))
    write_replay(
        Path(os.environ['TXGNN_MOLECULE_GAP_SOURCE']),
        Path(os.environ['TXGNN_MOLECULE_GAP_OUTPUT']),
        fixture=False,
        canonical_dir=Path(os.environ['TXGNN_MOLECULE_GAP_CANONICAL_SNAPSHOT']),
        canonical_manifest=Path(os.environ['TXGNN_MOLECULE_GAP_CANONICAL_MANIFEST']),
        chunksize=250_000,
        admission=admission,
    )
    verify_replay(Path(os.environ['TXGNN_MOLECULE_GAP_OUTPUT']))
else:
    print('SKIPPED: full replay is txgnn-worker-only and requires explicit environment paths')